# Pixeloza - demo

Osobny notebook, tylko do demonstracji - modele/generowanie bez zmian, poprawiony frontend (motyw pixel-art, własny układ, dekoracje) i uruchamia Gradio z 3 dropdownami.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q gradio x-transformers open_clip_torch

project_path = "/content/drive/MyDrive/Project"


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.4/143.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.8 MB/s eta 0:00:00


### KONFIGURACJA

In [2]:
WEIGHTS = {
    "vae_model": f"{project_path}/code/vae/model_VQ_VAE.pth",
    "vae_transformer": f"{project_path}/Trained_weights/with_clip/clip_transformer_vae.pt",
    "gan_vqgan": f"{project_path}/Trained_weights/with_clip/VQGAN.pt",
    "gan_prior": f"{project_path}/Trained_weights/with_clip/gan_token_prior.pt",
    "diffusion_original": f"{project_path}/Trained_weights/with_clip/dyfuzja_z_klipem_wagi.pth",
}

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import gradio as gr
import numpy as np
import math
import base64, io
from PIL import Image
from x_transformers import TransformerWrapper, Encoder

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Używamy: {device}")

Używamy: cuda


### CLIP

In [4]:
from transformers import CLIPTokenizer, CLIPTextModel

CLIP_NAME = "openai/clip-vit-base-patch32"
clip_tokenizer = CLIPTokenizer.from_pretrained(CLIP_NAME)
clip_text_model = CLIPTextModel.from_pretrained(CLIP_NAME).to(device)
clip_text_model.eval()
for p in clip_text_model.parameters():
    p.requires_grad = False

TEXT_DIM = clip_text_model.config.hidden_size
MAX_LEN = 32

@torch.no_grad()
def encode_text_tokens(captions):
    enc = clip_tokenizer(captions, padding="max_length", max_length=MAX_LEN,
                          truncation=True, return_tensors="pt").to(device)
    out = clip_text_model(**enc)
    return out.last_hidden_state, enc["attention_mask"]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

...

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

[transformers] CLIPTextModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
vision_model.post_layernorm.weight                             | UNEXPECTED |  | 
visual_projection.weight                                       | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.l

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

In [5]:
from transformers import CLIPModel

clip_model_full = CLIPModel.from_pretrained(CLIP_NAME).to(device).eval()
for p in clip_model_full.parameters():
    p.requires_grad = False

@torch.no_grad()
def clip_embed_pooled(texts):
    enc = clip_tokenizer(texts, padding=True, truncation=True, max_length=77, return_tensors="pt").to(device)
    e = clip_model_full.get_text_features(**enc)

    # Jeśli e jest obiektem BaseModelOutputWithPooling, wyciągamy z niego tensor
    if not isinstance(e, torch.Tensor):
        e = getattr(e, "text_embeds", getattr(e, "pooler_output", e[0]))

    return F.normalize(e.float(), dim=-1)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

## Model VQ-VAE

In [6]:
class VectorQuantizer(torch.nn.Module):
    def __init__(self, num_embeddings, embedding_dim):
        super().__init__()
        self.embeddings = torch.nn.Embedding(num_embeddings, embedding_dim)

class VQVAE(torch.nn.Module):
    def __init__(
        self, in_channels=4, hidden_dim=128, num_embeddings=64, embedding_dim=64
    ):
        super().__init__()
        self.vq_vae = VectorQuantizer(num_embeddings, embedding_dim)
        self.decoder = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(
                embedding_dim, hidden_dim, kernel_size=3, stride=1, padding=1
            ),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(
                hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1
            ),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(
                hidden_dim // 2, in_channels, kernel_size=4, stride=2, padding=1
            ),
            torch.nn.Sigmoid(),
        )

## Model VQ-GAN

In [7]:
class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=256, embedding_dim=32, commitment_cost=0.5):
        super().__init__()
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost
        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)

    def forward(self, z_e):
        B, C, H, W = z_e.shape
        z_e_flat = z_e.permute(0, 2, 3, 1).reshape(-1, C)
        distances = (
            z_e_flat.pow(2).sum(dim=1, keepdim=True)
            - 2 * z_e_flat @ self.embeddings.weight.t()
            + self.embeddings.weight.pow(2).sum(dim=1)
        )
        indices = distances.argmin(dim=1)
        z_q = self.embeddings(indices).view(B, H, W, C).permute(0, 3, 1, 2)
        return z_q, 0, indices.view(B, H, W)


class VQGAN(nn.Module):
    def __init__(self, in_channels=4, hidden_dim=64, latent_dim=32, num_embeddings=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(in_channels, hidden_dim, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(hidden_dim, latent_dim, 3, stride=1, padding=1),
        )
        self.vq_vae = VectorQuantizer(num_embeddings=num_embeddings, embedding_dim=latent_dim)
        self.decoder = nn.Sequential(
            nn.Conv2d(latent_dim, hidden_dim, 3, stride=1, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, hidden_dim, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(hidden_dim, in_channels, 4, stride=2, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z_e = self.encoder(x)
        z_q, vq_loss, indices = self.vq_vae(z_e)
        x_recon = self.decoder(z_q)
        return x_recon, vq_loss, indices

class TokenPrior(nn.Module):
    def __init__(self, vocab=256, seq_len=64, cond_dim=512, d=256, layers=6, heads=4):
        super().__init__()
        self.vocab, self.seq_len = vocab, seq_len
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Parameter(torch.randn(1, seq_len, d) * 0.02)
        self.cond_proj = nn.Sequential(nn.Linear(cond_dim, d), nn.GELU(), nn.Linear(d, d))
        self.null = nn.Parameter(torch.zeros(1, d))
        layer = nn.TransformerEncoderLayer(
            d, heads, 4 * d, dropout=0.1, activation="gelu", batch_first=True, norm_first=True
        )
        self.blocks = nn.TransformerEncoder(layer, layers, enable_nested_tensor=False)
        self.norm = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab)
        self.register_buffer("causal", torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), 1))

    def forward(self, cond, prefix, uncond=False):
        B = cond.shape[0]
        c = self.cond_proj(cond)
        if uncond:
            c = self.null.expand(B, -1)
        x = torch.cat([c[:, None, :], self.tok(prefix)], dim=1)
        T = x.shape[1]
        x = x + self.pos[:, :T]
        x = self.blocks(x, mask=self.causal[:T, :T])
        return self.head(self.norm(x))

## Model Diffusion original i new

In [8]:
class LinearNoiseScheduler:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cpu"):
        self.timesteps = timesteps
        self.device = device

        self.beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.alpha = 1.0 - self.beta
        self.alpha_cumprod = torch.cumprod(self.alpha, dim=0)

        self.alpha_cumprod_prev = torch.cat(
            [torch.ones(1, device=device), self.alpha_cumprod[:-1]]
        )

        self.posterior_variance = (
            self.beta * (1.0 - self.alpha_cumprod_prev) / (1.0 - self.alpha_cumprod)
        )

    def to(self, device):
        self.device = device
        for name in ["beta", "alpha", "alpha_cumprod", "alpha_cumprod_prev", "posterior_variance"]:
            setattr(self, name, getattr(self, name).to(device))
        return self

    def add_noise(self, x_0, t):
        #Dodaje szum do oryginalnego obrazu x_0 w kroku t
        noise = torch.randn_like(x_0)
        sqrt_alpha_cumprod = torch.sqrt(self.alpha_cumprod[t])[:, None, None, None]
        sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - self.alpha_cumprod[t])[:, None, None, None]

        x_t = sqrt_alpha_cumprod * x_0 + sqrt_one_minus_alpha_cumprod * noise
        return x_t, noise

class SimpleBlock(nn.Module):
    def __init__(self, in_c, out_c, time_emb_dim):
        super().__init__()
        self.time_mlp = nn.Linear(time_emb_dim, out_c)
        self.conv1 = nn.Conv2d(in_c, out_c, 3, padding=1)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.bn2 = nn.BatchNorm2d(out_c)

    def forward(self, x, t_emb):
        h = F.relu(self.bn1(self.conv1(x)))
        t_emb = F.relu(self.time_mlp(t_emb))[:, :, None, None]
        h = h + t_emb
        h = F.relu(self.bn2(self.conv2(h)))
        return h

class UNet32(nn.Module):
    def __init__(self, text_dim, in_channels=4, time_emb_dim=128, base=64):
        super().__init__()
        self.in_channels = in_channels
        c = base
        self.time_emb = nn.Embedding(1000, time_emb_dim)
        self.time_mlp = nn.Sequential(
            nn.Linear(time_emb_dim, time_emb_dim),
            nn.GELU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        self.cond_mlp = nn.Sequential(
            nn.Linear(text_dim, time_emb_dim),
            nn.GELU(),
            nn.Linear(time_emb_dim, time_emb_dim),
        )

        # Enkoder
        self.b1 = SimpleBlock(in_channels, c, time_emb_dim)        # [B, c, 32, 32]
        self.down1 = nn.MaxPool2d(2)
        self.b2 = SimpleBlock(c, 2 * c, time_emb_dim)              # [B, 2c, 16, 16]
        self.down2 = nn.MaxPool2d(2)

        # Wezel srodkowy
        self.mid = SimpleBlock(2 * c, 2 * c, time_emb_dim)         # [B, 2c, 8, 8]

        # Dekoder
        self.up2 = nn.ConvTranspose2d(2 * c, c, 2, stride=2)
        self.b3 = SimpleBlock(3 * c, c, time_emb_dim)              # c + 2c
        self.up1 = nn.ConvTranspose2d(c, c // 2, 2, stride=2)
        self.b4 = SimpleBlock(c // 2 + c, c // 2, time_emb_dim)

        self.final_conv = nn.Conv2d(c // 2, in_channels, 1)

    def forward(self, x, t, text_tokens, text_mask):
        mask_f = text_mask.unsqueeze(-1).float()
        pooled = (text_tokens * mask_f).sum(1) / mask_f.sum(1).clamp(min=1e-6)

        t_emb = self.time_mlp(self.time_emb(t)) + self.cond_mlp(pooled)

        x1 = self.b1(x, t_emb)
        x2 = self.b2(self.down1(x1), t_emb)

        mid = self.mid(self.down2(x2), t_emb)

        x3 = self.b3(torch.cat([self.up2(mid), x2], dim=1), t_emb)
        x4 = self.b4(torch.cat([self.up1(x3), x1], dim=1), t_emb)

        return self.final_conv(x4)

@torch.no_grad()
def denoise_from(model, scheduler, x, t_start, text_tokens, text_mask):
    model.eval()
    n = x.shape[0]

    for i in reversed(range(t_start + 1)):
        t = torch.full((n,), i, device=x.device, dtype=torch.long)
        predicted_noise = model(x, t, text_tokens, text_mask)

        alpha = scheduler.alpha[i]
        alpha_cumprod = scheduler.alpha_cumprod[i]

        x = (1 / torch.sqrt(alpha)) * (
            x - ((1 - alpha) / torch.sqrt(1 - alpha_cumprod)) * predicted_noise
        )

        if i > 0:
            sigma = torch.sqrt(scheduler.posterior_variance[i])
            x = x + sigma * torch.randn_like(x)

    return x


@torch.no_grad()
def sample_ddpm(model, scheduler, text_tokens, text_mask):
    x = torch.randn(text_tokens.shape[0], model.in_channels, 32, 32, device=device)
    x = denoise_from(model, scheduler, x, scheduler.timesteps - 1, text_tokens, text_mask)
    return (x.clamp(-1, 1) + 1) / 2  # Przekształcenie zakresu [-1, 1] -> [0, 1]

## Rejestr modeli - leniwe wczytywanie (dopiero przy pierwszym użyciu)

In [9]:
import open_clip
_cache = {}

def _to_pil(img_np):
    img_uint8 = (img_np[..., :4] * 255).astype(np.uint8)
    return Image.fromarray(img_uint8, mode="RGBA").resize((256, 256), Image.NEAREST)

# _______________________________________________________________
# VAE

def load_open_clip_vae():
  if "open_clip_vae" not in _cache:
    oc_model, _, _ = open_clip.create_model_and_transforms(
        "ViT-B-32", pretrained="openai"
    )
    oc_model = oc_model.to(device).eval()
    for p in oc_model.parameters():
      p.requires_grad = False
    _cache["open_clip_vae"] = oc_model
  return _cache["open_clip_vae"]

def _load_vq_masked_transformer_clip(path, dim=256, depth=6, heads=8):
    checkpoint = torch.load(path, map_location=device)

    # max_seq_len ustawiony poprawnie na 64 (dla siatki 8x8)
    net = TransformerWrapper(
        num_tokens=checkpoint["vocab_size"],
        max_seq_len=64,
        attn_layers=Encoder(
            dim=dim, depth=depth, heads=heads, attn_flash=True, cross_attend=True
        ),
    ).to(device)
    net.load_state_dict(checkpoint["transformer_state_dict"])
    net.eval()

    text_proj = nn.Linear(TEXT_DIM, dim).to(device)
    text_proj.load_state_dict(checkpoint["text_projection_state_dict"])
    text_proj.eval()

    return net, text_proj, checkpoint["mask_token_id"]


@torch.no_grad()
def _generate_vq_clip(
    vq_model, transformer_net, text_proj, mask_token_id, prompt, iterations=10
):
    # 1. Kodowanie tekstu przez pooled CLIP embedding -> shape [1, 512] -> proj -> [1, 1, 256]
    oc_model = load_open_clip_vae()
    text_tokens = open_clip.tokenize([prompt]).to(device)
    text_features = oc_model.encode_text(text_tokens)
    condition_context = text_proj(text_features).unsqueeze(1)

    # 2. Inicjalizacja sekwencji maskami
    seq_len = 64
    tokens = torch.full(
        (1, seq_len), mask_token_id, dtype=torch.long, device=device
    )

    # 3. Pętla samplowania zgodna z działającym skryptem
    for _ in range(iterations):
        logits = transformer_net(tokens, context=condition_context)
        probs = F.softmax(logits, dim=-1)
        tokens = torch.argmax(probs, dim=-1)

    # 4. Przycięcie tokenów do słownika VAE (0-63)
    image_tokens = tokens.clamp(0, 63)

    # 5. Rekonstrukcja obrazu
    z_q = vq_model.vq_vae.embeddings(image_tokens)
    z_q = z_q.view(1, 8, 8, 64).permute(0, 3, 1, 2).contiguous()
    generated_image = vq_model.decoder(z_q)

    # Formatowanie pod numpy -> PIL
    img_np = generated_image[0].permute(1, 2, 0).clamp(0, 1).cpu().numpy()
    return img_np


def load_vae():
    if "vae" not in _cache:
        m = VQVAE(num_embeddings=64).to(device)
        m.load_state_dict(
            torch.load(WEIGHTS["vae_model"], map_location=device), strict=False
        )
        m.eval()

        net, text_proj, mask_id = _load_vq_masked_transformer_clip(
            WEIGHTS["vae_transformer"]
        )
        _cache["vae"] = (m, net, text_proj, mask_id)
    return _cache["vae"]


def generate_vae(prompt):
    m, net, text_proj, mask_id = load_vae()
    img_np = _generate_vq_clip(m, net, text_proj, mask_id, prompt)
    return _to_pil(img_np)

# _______________________________________________________________
# GAN original

@torch.no_grad()
def generate_tokens(prior, prompt, n=1, temperature=1.0, top_k=50, guidance=3.0):
    prior.eval()

    # 1. Zmieniono clip_embed na zdefiniowane wcześniej clip_embed_pooled
    cond = clip_embed_pooled([prompt]).repeat(n, 1)  # [n, 512]
    seq = torch.zeros(n, 0, dtype=torch.long, device=device)

    for _ in range(prior.seq_len):
        # Generowanie logitów warunkowych
        logits_cond = prior(cond, seq)[:, -1]

        # Classifier-Free Guidance
        if guidance != 1.0:
            logits_uncond = prior(cond, seq, uncond=True)[:, -1]
            logits = logits_uncond + guidance * (logits_cond - logits_uncond)
        else:
            logits = logits_cond

        # Dzielenie przez temperaturę z zabezpieczeniem przed 0
        if temperature > 0:
            logits = logits / temperature

        # Top-K filtering
        if top_k and top_k < logits.shape[-1]:
            kth = logits.topk(top_k, dim=-1).values[:, -1:]
            logits = logits.masked_fill(logits < kth, float("-inf"))

        # Sampling kolejnego tokenu
        probs = F.softmax(logits, dim=-1)
        nxt = torch.multinomial(probs, 1)
        seq = torch.cat([seq, nxt], dim=1)

    return seq  # [n, seq_len]


# --- 1. ŁADOWANIE MODELU ---
def load_gan_original():
    if "gan_original" not in _cache:
        model = VQGAN(in_channels=4, hidden_dim=64, latent_dim=32, num_embeddings=256).to(device)
        model.load_state_dict(torch.load(WEIGHTS["gan_vqgan"], map_location=device))
        model.eval()

        prior = TokenPrior(256).to(device)
        prior.load_state_dict(torch.load(WEIGHTS["gan_prior"], map_location=device))
        prior.eval()

        _cache["gan_original"] = (model, prior)
    return _cache["gan_original"]


@torch.no_grad()
def generate_gan_original(prompt, temperature=1.0, top_k=50, guidance=2.0):
    m, prior = load_gan_original()

    # Generujemy tokeny za pomocą Priora
    idx = generate_tokens(prior, prompt, n=1, temperature=temperature, top_k=top_k, guidance=guidance)

    # Dekodujemy siatkę tokenów do obrazu przez VQGAN
    z_q = m.vq_vae.embeddings(idx.view(-1, 8, 8)).permute(0, 3, 1, 2)
    out_tensor = m.decoder(z_q).clamp(0, 1)

    img_np = out_tensor[0].permute(1, 2, 0).cpu().numpy()
    return _to_pil(img_np)

# _______________________________________________________________
# Diffusion original
def load_diffusion_original():
    if "diff_orig" not in _cache:
        scheduler = LinearNoiseScheduler(timesteps=1000).to(device)
        model = UNet32(text_dim=TEXT_DIM, in_channels=4, time_emb_dim=128, base=64).to(device)
        model.load_state_dict(torch.load(WEIGHTS["diffusion_original"], map_location=device))
        model.eval()
        _cache["diff_orig"] = (model, scheduler)  # poprawiono zmienne
    return _cache["diff_orig"]


@torch.no_grad()
def generate_diffusion_original(prompt):
    m, sched = load_diffusion_original()

    # Kodujemy 1 prompt
    tokens, masks = encode_text_tokens([prompt])

    # Generujemy obraz za pomocą wspólnego sample_ddpm
    out_tensor = sample_ddpm(m, sched, tokens, masks)

    img_np = out_tensor[0].permute(1, 2, 0).cpu().numpy()
    return _to_pil(img_np)


MODEL_REGISTRY = {
    "VAE": generate_vae,
    "GAN (original)": generate_gan_original,
    "Diffusion (original)": generate_diffusion_original
}

## Frontend

Własny CSS - `image-rendering: pixelated` na galerii, żeby 32x32 nie rozmywało się przy powiekszeniu.

In [10]:
# MASCOT = "iVBORw0KGgoAAAANSUhEUgAAAMAAAADACAYAAABS3GwHAAACkUlEQVR4nO3dsU0DQRBAUR+iGhKaoBLqIKQOKqEJEtoxNazMam/vvxdb1srS1ySj8XHb3Mfv/b76DWWfL8ex+g2PeFr9AFhJAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKQJgLTT7XHY7bm2s+0OmQCkCYA0AZAmANIEQJoASBMAaQIgTQCkCYA0AZA2fS/Dbg+PmL07ZAKQJgDSBECaAEgTAGkCIE0ApAmANAGQJgDSBEDa8J6F3R7ObHR3yAQgTQCkCYA0AZAmANIEQJoASBMAaQIgTQCkCYA0AZAmANIEQJoASBMAaQIgTQCkCYA0AZAmANIEQJoASHte/YBHjd6BmX3XyHv2YgKQJgDSBECaAEgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGkCIE0ApA3djLndendj2MvoHSQTgDQBkCYA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0rb/j7Cvt9ehz79//0x6yTXUfk8TgDQBkCYA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0twF4lLcBYIBAiBNAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKRtfxdottHdklF2q9YyAUgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGkCIM0u0GKju0Z2h/6XCUCaAEgTAGkCIE0ApAmANAGQJgDSBECaAEgTAGnTd4Fm77qcbZdm9vtH1X7/USYAaQIgTQCkCYA0AZAmANIEQJoASBMAaQIgTQCkne4u0O67K7Xvn232e0wA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKQJgDQBkCYA0gRAmgBIEwBpAiBNAKQJAAAAAAAAAAAAgO39Ae+qWM9L+V5VAAAAAElFTkSuQmCC"
# BACKGROUND = "iVBORw0KGgoAAAANSUhEUgAAAEAAAABACAIAAAAlC+aJAAAAfElEQVR4nO3YsQ3CABAEQUBkJnIvTlwL/YeOcE4V75WlmQJOWn32z337Pib9zmN0/zW6fgEBNQE1ATUBNQE1ATUBNQE1ATUBNQE1ATUBNQE1AbX39P/+s6yj+7e/gICagJqAmoCagJqAmoCagJqAmoCagJqAmoCagJqA2h/6vgckrTIh5gAAAABJRU5ErkJggg=="
import base64

def load_b64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("ascii")

MASCOT = load_b64(f"{project_path}/data/mascot.png")
BACKGROUND = load_b64(f"{project_path}/data/background.png")


CUSTOM_CSS = f"""
@import url('https://fonts.googleapis.com/css2?family=Press+Start+2P&display=swap');

:root {{
    --bg-deep: #1a1530;
    --bg-panel: #241d3f;
    --accent-cyan: #7cd6ff;
    --accent-blue: #e5dbfecc;
    --accent-yellow: #ffe066;
    --text: #7b7b7b;
}}

.gradio-container {{
    background: var(--bg-deep) !important;
    background-image: url("data:image/png;base64,{BACKGROUND}") !important;
    background-size: cover !important;
    background-position: center !important;
    background-repeat: no-repeat !important;
    background-attachment: fixed !important;
    font-family: 'Press Start 2P', monospace !important;
    color: var(--text) !important;
}}

#pixeloza-header {{
    display: flex;
    align-items: center;
    justify-content: center;
    gap: 20px;
    padding: 18px 10px;
    margin-bottom: 10px;
    background: var(--bg-panel);
    border: 4px solid var(--accent-cyan);
    box-shadow: 6px 6px 0 rgba(0,0,0,0.4);
}}

#pixeloza-header img {{
    image-rendering: pixelated;
    width: 128px;
    height: 128px;
}}

#pixeloza-header h1 {{
    font-family: 'Press Start 2P', monospace;
    font-size: 22px;
    color: var(--accent-yellow);
    text-shadow: 3px 3px 0 #000;
    margin: 0;
}}

#pixeloza-header p {{
    font-family: monospace;
    font-size: 13px;
    color: var(--accent-cyan);
    margin: 4px 0 0 0;
}}

.gr-panel, .gr-box, .gr-form {{
    background: var(--bg-panel) !important;
    border: 3px solid var(--accent-blue) !important;
    box-shadow: 4px 4px 0 rgba(0,0,0,0.35) !important;
}}

/* prostokat z lewej (dropdowny + checkboxy) */
#controls-column {{
    background: rgba(94, 60, 177, 0.8) !important;
    border: 3px solid var(--accent-blue) !important;
    padding: 12px !important;
}}

/* prostokat z prawej (galeria wynikow) */
#output-column {{
    background: rgba(94, 60, 177, 0.8) !important;
    border: 3px solid var(--accent-blue) !important;
}}

/* tlo i tekst kazdego dropdowna */
#glasses-dropdown, #head-dropdown, #body-dropdown {{
    background: rgba(94, 60, 177, 0.8) !important;
}}
#glasses-dropdown input, #head-dropdown input, #body-dropdown input,
#glasses-dropdown span, #head-dropdown span, #body-dropdown span {{
    color: rgba(0, 0, 0, 0.8) !important;
}}

/* caly blok checkboxow (tlo calego "pudelka") */
#model-checkboxes {{
    background: rgba(94, 60, 177, 0.8) !important;
}}
/* kazdy pojedynczy checkbox z osobna - i jego tlo, i tekst */
#model-checkboxes label {{
    background: rgba(94, 60, 177, 0.8) !important;
    color: rgba(0, 0, 0, 0.8) !important;
    border: 2px solid var(--accent-blue) !important;
}}

button.primary {{
    font-family: 'Press Start 2P', monospace !important;
    background: var(--accent-blue) !important;
    color: #1a1530 !important;
    border: 3px solid #000 !important;
    box-shadow: 4px 4px 0 #000 !important;
    font-size: 12px !important;
}}

button.primary:hover {{
    transform: translate(2px, 2px);
    box-shadow: 2px 2px 0 #000 !important;
}}

#output-gallery img {{
    image-rendering: pixelated !important;
}}
"""

In [11]:
TEXT_MODELS = {"Diffusion (original)", "GAN (original)", "VAE"}

def generate_compare(prompt, selected_models):
    results = []
    for name in selected_models:
        try:
            if name in TEXT_MODELS:
                img = MODEL_REGISTRY[name](prompt)
            else:
                print(f"Ostrzeżenie: {name} nie obsługuje tekstu jeszcze. Używam (0, 0, 0).")
                img = MODEL_REGISTRY[name](0, 0, 0)
            results.append((img, name))
        except Exception as e:
            print(f"Błąd przy {name}: {e}")
            results.append((Image.new("RGBA", (256, 256), (255, 0, 0, 128)), f"{name} - BŁĄD"))
    return results


with gr.Blocks(css=CUSTOM_CSS, title="Pixeloza") as demo:
    gr.HTML(f"""
    <div id="pixeloza-header">
        <img src="data:image/png;base64,{MASCOT}" alt="maskotka"/>
        <div>
            <h1>PIXELOZA</h1>
            <p>Wpisz prompt &gt; Zaznacz modele &gt; Porównaj wyniki</p>
        </div>
    </div>
    """)

    with gr.Row(elem_id="main-row"):
        with gr.Column(scale=1, elem_id="controls-column"):
            prompt_input = gr.Textbox(label="Opisz awatar (po angielsku)", placeholder="np. a character with green glasses", value="a character with square dark gray glasses, a void-shaped head and a teal-colored body")
            models_cbg = gr.CheckboxGroup(choices=list(MODEL_REGISTRY.keys()), label="Modele do porównania", value=["Diffusion (original)"], elem_id="model-checkboxes")
            generate_btn = gr.Button("Generuj", variant="primary")

        with gr.Column(scale=2):
            gallery = gr.Gallery(label="Wyniki", elem_id="output-gallery", columns=5, height="auto", object_fit="contain")


    generate_btn.click(fn=generate_compare, inputs=[prompt_input, models_cbg], outputs=gallery)

demo.launch(share=True)

/tmp/ipykernel_721/1586717896.py:19: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Pixeloza") as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1ddbeb0b0f396b2201.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
TEXT_MODELS = {"VAE"}

def generate_compare(prompt, selected_models):
    results = []
    for name in selected_models:
        try:
            if name in TEXT_MODELS:
                img = MODEL_REGISTRY[name](prompt)
            else:
                print(f"Ostrzeżenie: {name} nie obsługuje tekstu jeszcze. Używam (0, 0, 0).")
                img = MODEL_REGISTRY[name](0, 0, 0)
            results.append((img, name))
        except Exception as e:
            print(f"Błąd przy {name}: {e}")
            results.append((Image.new("RGBA", (256, 256), (255, 0, 0, 128)), f"{name} - BŁĄD"))
    return results

generate_compare("a character with green glasses", {"Diffusion (original)", "GAN (original)", "VAE"})

Ostrzeżenie: GAN (original) nie obsługuje tekstu jeszcze. Używam (0, 0, 0).
Błąd przy GAN (original): text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).


open_clip_model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

open_clip_model.safetensors: downloading bytes:           |  0.00B            

/usr/local/lib/python3.13/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


Ostrzeżenie: Diffusion (original) nie obsługuje tekstu jeszcze. Używam (0, 0, 0).
Błąd przy Diffusion (original): generate_diffusion_original() takes 1 positional argument but 3 were given


/tmp/ipykernel_1551/3643812182.py:6: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  return Image.fromarray(img_uint8, mode="RGBA").resize((256, 256), Image.NEAREST)


[(<PIL.Image.Image image mode=RGBA size=256x256>, 'GAN (original) - BŁĄD'),
 (<PIL.Image.Image image mode=RGBA size=256x256>, 'VAE'),
 (<PIL.Image.Image image mode=RGBA size=256x256>,
  'Diffusion (original) - BŁĄD')]